# 5. Préparation des données pour la modélisation

Après le Feature Engineering, nous préparons les données pour leur utilisation dans les algorithmes de Machine Learning. Cette étape consiste à définir la variable cible, sélectionner les variables explicatives, séparer les données en ensembles d’apprentissage et de test, puis construire un prétraitement adapté aux différents types de variables.

Les transformations qui nécessitent un apprentissage à partir des données seront ajustées uniquement sur l’ensemble d’apprentissage afin de limiter les risques de fuite de données.

### 5.1. Chargement des données

Nous chargeons le fichier produit lors du Feature Engineering. Nous vérifions ses dimensions et son contenu avant de poursuivre la préparation.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Chargement des données transformées
chemin_donnees = Path(
    "../data/interim/bank_marketing_feature_engineered.csv"
)

df = pd.read_csv(chemin_donnees, sep=";")

print("Dimensions du jeu de données :", df.shape)
display(df.head())

Dimensions du jeu de données : (4119, 21)


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,jamais_contacte
0,30,worker,married,basic,no,yes,no,cellular,spring,fri,...,2,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no,1
1,39,worker,single,high.school,no,no,no,telephone,spring,fri,...,4,0,nonexistent,1.1,93.994,-36.4,4.855,5191.0,no,1
2,25,worker,married,high.school,no,yes,no,telephone,summer,wed,...,1,0,nonexistent,1.4,94.465,-41.8,4.962,5228.1,no,1
3,38,worker,married,basic,no,no,no,telephone,summer,fri,...,3,0,nonexistent,1.4,94.465,-41.8,4.959,5228.1,no,1
4,47,white_collar,married,university.degree,no,yes,no,cellular,nov,mon,...,1,0,nonexistent,-0.1,93.200,-42.0,4.191,5195.8,no,1


### 5.2. Vérification du jeu de données

Avant de construire les ensembles d’apprentissage et de test, nous contrôlons les dimensions, les types de variables, les valeurs manquantes et la présence de doublons. Ces vérifications permettent de repérer les problèmes qui pourraient perturber la modélisation.

In [2]:
# Dimensions et types
print("Dimensions :", df.shape)

print("\nTypes des variables :")
print(df.dtypes)

# Valeurs manquantes
print("\nValeurs manquantes par variable :")
print(df.isna().sum().sort_values(ascending=False).head(10))

# Doublons
print("\nNombre de lignes dupliquées :", df.duplicated().sum())

# Vérification de la cible
print("\nRépartition de la cible :")
print(df["y"].value_counts(dropna=False))

Dimensions : (4119, 21)

Types des variables :
age                  int64
job                    str
marital                str
education              str
default                str
housing                str
loan                   str
contact                str
month                  str
day_of_week            str
duration             int64
campaign             int64
previous             int64
poutcome               str
emp.var.rate       float64
cons.price.idx     float64
cons.conf.idx      float64
euribor3m          float64
nr.employed        float64
y                      str
jamais_contacte      int64
dtype: object

Valeurs manquantes par variable :
age            0
job            0
marital        0
education      0
default        0
housing        0
loan           0
contact        0
month          0
day_of_week    0
dtype: int64

Nombre de lignes dupliquées : 0

Répartition de la cible :
y
no     3668
yes     451
Name: count, dtype: int64


### 5.3. Définition de la cible et des variables explicatives

La variable `y` indique si le client a souscrit à un dépôt à terme. Elle constitue la cible à prédire. Les autres variables servent à expliquer cette souscription.

Nous écartons `duration`, car la durée de l’appel n’est connue qu’à la fin de celui-ci. L’utiliser pour cibler les clients avant leur appel introduirait une fuite de données. Nous écartons également `day_of_week`, conformément au modèle de référence. La variable `pdays`, si elle est encore présente, est remplacée par l’indicateur `jamais_contacte`.

In [3]:
# Définition de la variable cible
target = "y"

# Vérification de la présence de la cible
if target not in df.columns:
    raise ValueError(f"La variable cible {target!r} est absente.")

# Variables à exclure des entrées du modèle
variables_a_exclure = [
    "duration",
    "day_of_week",
    "pdays"
]

# On ne conserve que les variables d'exclusion présentes
variables_a_exclure = [
    col for col in variables_a_exclure
    if col in df.columns
]

# Construction des variables explicatives et de la cible
X = df.drop(columns=[target] + variables_a_exclure)
y = df[target]

print("Variables exclues :", variables_a_exclure)
print("Dimensions de X :", X.shape)
print("Dimensions de y :", y.shape)

print("\nRépartition de la cible en pourcentage :")
print(y.value_counts(normalize=True).mul(100).round(2))

Variables exclues : ['duration', 'day_of_week']
Dimensions de X : (4119, 18)
Dimensions de y : (4119,)

Répartition de la cible en pourcentage :
y
no     89.05
yes    10.95
Name: proportion, dtype: float64


### 5.4. Séparation des ensembles d’apprentissage et de test

Nous séparons les données en deux ensembles : 80 % pour l’apprentissage et 20 % pour le test. L’ensemble d’apprentissage servira à ajuster les transformations et à entraîner les modèles. L’ensemble de test sera conservé pour évaluer leur capacité à généraliser à des observations non utilisées pendant l’apprentissage.

Nous utilisons une séparation stratifiée afin de conserver, autant que possible, la même proportion de clients ayant souscrit et n’ayant pas souscrit dans les deux ensembles.

In [4]:
from sklearn.model_selection import train_test_split

# Séparation train/test avec conservation de la répartition de y
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test  :", X_test.shape)
print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test  :", y_test.shape)

print("\nRépartition de y_train (%) :")
print(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nRépartition de y_test (%) :")
print(y_test.value_counts(normalize=True).mul(100).round(2))

Dimensions de X_train : (3295, 18)
Dimensions de X_test  : (824, 18)
Dimensions de y_train : (3295,)
Dimensions de y_test  : (824,)

Répartition de y_train (%) :
y
no     89.04
yes    10.96
Name: proportion, dtype: float64

Répartition de y_test (%) :
y
no     89.08
yes    10.92
Name: proportion, dtype: float64


### 5.5. Identification des variables numériques et catégorielles

Les variables explicatives ne sont pas toutes de même nature. Certaines sont numériques, comme `age` ou `campaign`, tandis que d’autres sont catégorielles, comme `job`, `education` ou `marital`.

Nous identifions ces deux groupes afin de leur appliquer un prétraitement adapté avant l’entraînement des modèles.

In [6]:
# Identification des variables numériques et catégorielles
colonnes_numeriques = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

colonnes_categorielles = X_train.select_dtypes(
    include=["str", "category", "bool"]
).columns.tolist()

print("Variables numériques :", colonnes_numeriques)
print("\nVariables catégorielles :", colonnes_categorielles)

print("\nNombre total de variables identifiées :",
      len(colonnes_numeriques) + len(colonnes_categorielles))

print("Nombre de colonnes dans X_train :", X_train.shape[1])

Variables numériques : ['age', 'campaign', 'previous', 'emp.var.rate', 'cons.price.idx', 'cons.conf.idx', 'euribor3m', 'nr.employed', 'jamais_contacte']

Variables catégorielles : ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

Nombre total de variables identifiées : 18
Nombre de colonnes dans X_train : 18


### 5.6. Construction du prétraitement

Les variables numériques et catégorielles nécessitent des traitements différents. Nous allons utiliser `StandardScaler` pour standardiser les variables numériques et `OneHotEncoder` pour convertir les catégories en colonnes numériques. Le `ColumnTransformer` permettra d’appliquer automatiquement chaque traitement aux bonnes variables.

Le prétraitement sera ajusté sur l’ensemble d’apprentissage, puis appliqué à l’ensemble de test sans réapprendre les transformations.

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Prétraitement des variables numériques
transformateur_numerique = StandardScaler()

# Prétraitement des variables catégorielles
transformateur_categoriel = OneHotEncoder(
    handle_unknown="ignore"
)

# Combinaison des traitements
preprocesseur = ColumnTransformer(
    transformers=[
        (
            "numerique",
            transformateur_numerique,
            colonnes_numeriques
        ),
        (
            "categoriel",
            transformateur_categoriel,
            colonnes_categorielles
        )
    ]
)

print("Préprocesseur construit avec succès.")

Préprocesseur construit avec succès.


### 5.7. Application du prétraitement

Nous ajustons les transformations sur l’ensemble d’apprentissage, puis nous les appliquons à l’ensemble de test. Cette démarche permet de conserver les paramètres appris sur le train et d’éviter que les données de test influencent le prétraitement.

Nous vérifions ensuite les dimensions des matrices obtenues afin de nous assurer que les deux ensembles ont subi les mêmes transformations.

In [8]:
# Ajustement du prétraitement sur le train uniquement
X_train_transforme = preprocesseur.fit_transform(X_train)

# Application du prétraitement au test
X_test_transforme = preprocesseur.transform(X_test)

print("Dimensions de X_train avant prétraitement :", X_train.shape)
print("Dimensions de X_train après prétraitement :", X_train_transforme.shape)

print("\nDimensions de X_test avant prétraitement :", X_test.shape)
print("Dimensions de X_test après prétraitement :", X_test_transforme.shape)

assert X_train_transforme.shape[1] == X_test_transforme.shape[1]

print("\nVérification réussie : les deux ensembles ont le même nombre de variables transformées.")

Dimensions de X_train avant prétraitement : (3295, 18)
Dimensions de X_train après prétraitement : (3295, 40)

Dimensions de X_test avant prétraitement : (824, 18)
Dimensions de X_test après prétraitement : (824, 40)

Vérification réussie : les deux ensembles ont le même nombre de variables transformées.


### 5.8. Contrôle des variables numériques transformées

Nous contrôlons les variables numériques après standardisation. Sur l’ensemble d’apprentissage, chaque variable standardisée doit avoir une moyenne proche de zéro et un écart-type proche de un, sauf dans certains cas particuliers, notamment lorsque la variable est constante.

In [9]:
# Récupération des variables numériques transformées
X_train_numerique = X_train_transforme[:, :len(colonnes_numeriques)]

# Calcul des moyennes et écarts-types sur le train
moyennes = np.asarray(X_train_numerique.mean(axis=0)).ravel()
ecarts_types = np.asarray(X_train_numerique.std(axis=0)).ravel()

controle_standardisation = pd.DataFrame({
    "variable": colonnes_numeriques,
    "moyenne_apres_standardisation": moyennes,
    "ecart_type_apres_standardisation": ecarts_types
})

display(controle_standardisation.round(4))

,variable,moyenne_apres_standardisation,ecart_type_apres_standardisation
0,age,-0.0,1.0
1,campaign,0.0,1.0
2,previous,-0.0,1.0
3,emp.var.rate,-0.0,1.0
4,cons.price.idx,-0.0,1.0
5,cons.conf.idx,0.0,1.0
6,euribor3m,0.0,1.0
7,nr.employed,0.0,1.0
8,jamais_contacte,-0.0,1.0


### 5.9. Contrôle de la répartition de la cible

La cible `y` est déséquilibrée : les clients qui souscrivent sont moins nombreux que ceux qui ne souscrivent pas. Nous vérifions que la séparation stratifiée a conservé des proportions comparables entre les ensembles d’apprentissage et de test.

Ce déséquilibre devra être pris en compte lors de l’évaluation des modèles, notamment en examinant le rappel, le F1-score et l’AUC, plutôt que de s’appuyer uniquement sur l’exactitude globale.

In [10]:
# Comparaison des répartitions de la cible
repartition_cible = pd.DataFrame({
    "train (%)": y_train.value_counts(normalize=True).mul(100),
    "test (%)": y_test.value_counts(normalize=True).mul(100)
}).round(2)

display(repartition_cible)

,train (%),test (%)
y,,
no,89.04,89.08
yes,10.96,10.92


### 5.10. Bilan des contrôles

Nous rassemblons les principaux contrôles réalisés au cours de cette étape : dimensions des ensembles, présence de valeurs manquantes dans la cible, cohérence du prétraitement et répartition de la cible. Ces vérifications permettent de confirmer que les données sont prêtes pour l’entraînement des modèles.

In [11]:
# Vérifications finales
print("Nombre d'observations dans le train :", len(X_train))
print("Nombre d'observations dans le test :", len(X_test))

print("\nValeurs manquantes dans y_train :", y_train.isna().sum())
print("Valeurs manquantes dans y_test :", y_test.isna().sum())

print("\nNombre de variables initiales :", X_train.shape[1])
print("Nombre de variables après prétraitement :", X_train_transforme.shape[1])

print("\nPrésence de valeurs non finies dans les variables numériques transformées :")
print(
    not np.isfinite(
        np.asarray(X_train_transforme[:, :len(colonnes_numeriques)])
    ).all()
)

assert len(X_train) + len(X_test) == len(df)
assert y_train.isna().sum() == 0
assert y_test.isna().sum() == 0
assert X_train_transforme.shape[1] == X_test_transforme.shape[1]

print("\nLes contrôles finaux sont terminés.")

Nombre d'observations dans le train : 3295
Nombre d'observations dans le test : 824

Valeurs manquantes dans y_train : 0
Valeurs manquantes dans y_test : 0

Nombre de variables initiales : 18
Nombre de variables après prétraitement : 40

Présence de valeurs non finies dans les variables numériques transformées :
False

Les contrôles finaux sont terminés.


### Conclusion

Les données ont été chargées, vérifiées et séparées en ensembles d’apprentissage et de test. La séparation stratifiée permet de conserver des proportions comparables de clients ayant souscrit et n’ayant pas souscrit.

Un prétraitement a été construit pour standardiser les variables numériques et encoder les variables catégorielles. Il a été ajusté uniquement sur l’ensemble d’apprentissage, puis appliqué à l’ensemble de test afin de limiter les risques de fuite de données.

Les données sont désormais prêtes pour l’étape suivante : la construction des modèles de prédiction, leur entraînement et leur évaluation à l’aide de métriques adaptées au déséquilibre de la cible.